# Beschreibung: 

# Importe:

In [70]:
import os

import pandas as pd
import numpy as np

from rst_functions import discretize, indiscernibility, dependency, quick_reduct, induce_rules

os.getcwd()

'/home/samel/01. Projekte/01. Master/Projekt/Kaggle_Daten'

# Daten laden:
Heart Failure Prediction Dataset: #https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset?select=healthcare-dataset-stroke-data.csv

In [71]:
heart = pd.read_csv("./health/Stroke_Prediction.csv")
print(heart.shape)

(5110, 12)


# Ausführung:

### Vorbereitung: (Datenaufbereitung)

In [72]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "stroke" # Macht aus dem Kontext heraus am meisten Sinn(Vor allem wenn es sich darum dreht)

In [73]:
cutoffs = {
    "Age": [30, 55, 65],      # jung <40, mittel, erhöht, 65<= hoch
}
cutoffs = {}

# Diskretisierung der numerischen Daten:
X = heart.drop(columns=[decision_attr])

# ALLE Konditionsattribute diskretisieren (inkl. kategoriale)
heart_disc = discretize(X, bins=7, cutoffs=cutoffs)

# Entscheidungsattribut wieder anhängen
heart_disc[decision_attr] = heart[decision_attr]
print(heart_disc.columns)

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'id_disc', 'age_disc', 'hypertension_disc',
       'heart_disease_disc', 'avg_glucose_level_disc', 'bmi_disc', 'stroke'],
      dtype='object')


In [74]:
cond_attrs = []
for col in heart_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte
        if col.endswith("_disc"):
            cond_attrs.append(col)
        # direkt kategorische Werte
        elif heart_disc[col].dtype == "object":
            cond_attrs.append(col)

cond_attrs.remove('id_disc') # Das ist ein Identifier, daher muss es raus.
print(cond_attrs)

['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'age_disc', 'hypertension_disc', 'heart_disease_disc', 'avg_glucose_level_disc', 'bmi_disc']


### Datenbetrachtung:

In [75]:
reduct, info = quick_reduct(heart_disc, cond_attrs, decision_attr)
rules = induce_rules(heart_disc, reduct, decision_attr)

γ(C) mit allen Attributen: 0.944227
Einzel-γ-Werte:
  gender: γ = 0.000196
  ever_married: γ = 0.000000
  work_type: γ = 0.004305
  Residence_type: γ = 0.000000
  smoking_status: γ = 0.000000
  age_disc: γ = 0.142857
  hypertension_disc: γ = 0.000000
  heart_disease_disc: γ = 0.000000
  avg_glucose_level_disc: γ = 0.000000
  bmi_disc: γ = 0.000000

Mindestens ein Attribut hat γ({a}) > 0 → benutze quick_reduct_monotone.


# Resultate

In [76]:
print("Ergebnisse:")

print(f"\n{decision_attr}:\n{reduct}")
print(f"\nAnzahl Regeln: {len(rules)}\n")

for r in rules[:10]:
    print(r)

#print("Rules:", rules_pass_biased[2]) # Einzeln
#print("Rules:", rules_pass_biased) # Das wären alle

Ergebnisse:

stroke:
['age_disc', 'bmi_disc', 'avg_glucose_level_disc', 'smoking_status', 'work_type', 'Residence_type', 'gender', 'hypertension_disc', 'ever_married', 'heart_disease_disc']

Anzahl Regeln: 3667

{'premise': {'age_disc': np.int64(5), 'bmi_disc': np.float64(5.0), 'avg_glucose_level_disc': np.int64(6), 'smoking_status': 'formerly smoked', 'work_type': 'Private', 'Residence_type': 'Urban', 'gender': 'Male', 'hypertension_disc': np.int64(0), 'ever_married': 'Yes', 'heart_disease_disc': np.int64(1)}, 'decision': np.int64(1), 'support': 1}
{'premise': {'age_disc': np.int64(5), 'bmi_disc': np.float64(5.0), 'avg_glucose_level_disc': np.int64(6), 'smoking_status': 'formerly smoked', 'work_type': 'Private', 'Residence_type': 'Rural', 'gender': 'Male', 'hypertension_disc': np.int64(0), 'ever_married': 'Yes', 'heart_disease_disc': np.int64(0)}, 'decision': np.int64(0), 'support': 2}
{'premise': {'age_disc': np.int64(5), 'bmi_disc': np.float64(5.0), 'avg_glucose_level_disc': np.int6

In [77]:
reduct, info = quick_reduct(heart_disc, cond_attrs, decision_attr, mode="interaction")
rules = induce_rules(heart_disc, reduct, decision_attr)

γ(C) mit allen Attributen: 0.944227
Einzel-γ-Werte:
  gender: γ = 0.000196
  ever_married: γ = 0.000000
  work_type: γ = 0.004305
  Residence_type: γ = 0.000000
  smoking_status: γ = 0.000000
  age_disc: γ = 0.142857
  hypertension_disc: γ = 0.000000
  heart_disease_disc: γ = 0.000000
  avg_glucose_level_disc: γ = 0.000000
  bmi_disc: γ = 0.000000

Erzwinge interaktionssensitive Variante (quick_reduct_interaction).


In [78]:
print("Ergebnisse:")

print(f"\n{decision_attr}:\n{reduct}")
print(f"\nAnzahl Regeln: {len(rules)}\n")

for r in rules[:10]:
    print(r)

#print("Rules:", rules_pass_biased[2]) # Einzeln
#print("Rules:", rules_pass_biased) # Das wären alle

Ergebnisse:

stroke:
['age_disc', 'bmi_disc', 'avg_glucose_level_disc', 'smoking_status', 'work_type', 'Residence_type', 'gender', 'hypertension_disc', 'ever_married', 'heart_disease_disc']

Anzahl Regeln: 3667

{'premise': {'age_disc': np.int64(5), 'bmi_disc': np.float64(5.0), 'avg_glucose_level_disc': np.int64(6), 'smoking_status': 'formerly smoked', 'work_type': 'Private', 'Residence_type': 'Urban', 'gender': 'Male', 'hypertension_disc': np.int64(0), 'ever_married': 'Yes', 'heart_disease_disc': np.int64(1)}, 'decision': np.int64(1), 'support': 1}
{'premise': {'age_disc': np.int64(5), 'bmi_disc': np.float64(5.0), 'avg_glucose_level_disc': np.int64(6), 'smoking_status': 'formerly smoked', 'work_type': 'Private', 'Residence_type': 'Rural', 'gender': 'Male', 'hypertension_disc': np.int64(0), 'ever_married': 'Yes', 'heart_disease_disc': np.int64(0)}, 'decision': np.int64(0), 'support': 2}
{'premise': {'age_disc': np.int64(5), 'bmi_disc': np.float64(5.0), 'avg_glucose_level_disc': np.int6